# Generación del dataset de torques con WHISKiT Physics (Tarea A: localización de un poste)

Este cuaderno compila el simulador **WHISKiT Physics** (Zweifel et al., 2021) en Colab y genera un dataset de señales de torque y fuerza en la base de los 27 bigotes derechos de la rata, para distintas posiciones de un poste fijo. La etiqueta de cada simulación es la posición exacta (x, y) del poste en el marco de la cabeza (+x adelante, +y derecha, en mm).

**Cómo usarlo**
1. Ejecuta las celdas en orden. La compilación tarda unos 3 minutos.
2. La celda de generación es reanudable: si Colab se desconecta, vuelve a ejecutarla y continúa donde quedó (cada simulación terminada deja una marca `DONE` en Drive).
3. Al final se crea `dataset_A.npz` en tu Drive.

Costo aproximado: ~3 min de CPU por segundo simulado por simulación. Con los 2 núcleos de Colab gratuito, 300 simulaciones de 1 s ≈ 7–8 h (en varias sesiones). Con más núcleos escala linealmente (`--jobs`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/whiskit_dataset'   # todo lo persistente vive aquí
os.makedirs(WORK, exist_ok=True)
print('Carpeta de trabajo:', WORK)

## 1. Dependencias y compilación del simulador

In [ ]:
%%bash
set -e
apt-get install -y -q libboost-program-options-dev libboost-filesystem-dev libboost-system-dev freeglut3-dev libgl1-mesa-dev libglu1-mesa-dev > /dev/null
cd /content
if [ ! -d whiskitphysics ]; then git clone --depth 1 https://github.com/SeNSE-lab/whiskitphysics.git; fi
echo "repositorio listo" 

In [ ]:
# Parche: añade la opción --PEG_LOC x y z al ejecutable (en el original la posición del poste está fija en el código).
p = "/content/whiskitphysics/code/src/main.cpp"; s = open(p).read()
if "PEG_LOC" not in s:
    s = s.replace('("SPEED", po::value<float>(&param->PEG_SPEED), "peg speed")',
      '("SPEED", po::value<float>(&param->PEG_SPEED), "peg speed")\n\t\t("PEG_LOC", po::value<std::vector<std::string> >()->multitoken(), "peg location x y z (mm)")')
    s = s.replace('\t\t\tstd::vector<std::string> angles;',
      '\t\t\tstd::vector<std::string> pegloc;\n\t\t\tif (vm.count("PEG_LOC") && (pegloc = vm["PEG_LOC"].as<std::vector<std::string> >()).size() == 3) {\n\t\t\t\tparam->PEG_LOC = btVector3(lexical_cast<float>(pegloc[0]), lexical_cast<float>(pegloc[1]), lexical_cast<float>(pegloc[2]));\n\t\t\t}\n\n\t\t\tstd::vector<std::string> angles;', 1)
    open(p, "w").write(s)
print("parche aplicado" if "PEG_LOC" in open(p).read() else "ERROR: parche no aplicado")

In [ ]:
%%bash
set -e
cd /content/whiskitphysics/code && mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release > /dev/null 2>&1
make -j$(nproc) whiskit 2>&1 | grep -E "error|Built target whiskit"
./whiskit --help | grep PEG_LOC && echo "COMPILADO OK" 

## 2. Scripts del generador y del empaquetador
Son los mismos que se usaron en el piloto; se escriben a disco tal cual para que el proceso sea idéntico.

In [ ]:
import os
os.makedirs('/content/dataset', exist_ok=True)
open('/content/dataset/run_dataset.py','w').write(r'''#!/usr/bin/env python3
"""
Generador de dataset para la Tarea A (localización de un poste a partir del torque).

Qué hace, paso a paso:
 1. Muestrea N posiciones (x, y) del poste dentro del alcance de la matriz derecha de bigotes.
 2. Lanza WHISKiT Physics para cada posición (poste fijo, batido activo, 27 bigotes derechos).
 3. Guarda cada simulación en su carpeta y escribe un manifiesto CSV con la etiqueta exacta.
 4. Corre varias simulaciones en paralelo (una por núcleo) y se puede reanudar si se interrumpe.

Uso:
   python3 run_dataset.py --n 40 --seed 0 --out pilot_A --time 1.0 --jobs 2
"""
import argparse, csv, os, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
CODE = os.path.abspath(os.path.join(HERE, "..", "whiskitphysics", "code"))
BIN = os.path.join(CODE, "build", "whiskit")
SCRIPTS = os.path.join(CODE, "scripts")          # whiskit resuelve rutas relativas desde aquí

# Rango de posiciones del poste (mm, marco de la cabeza: +x adelante, +y derecha, z altura)
X_RANGE = (12.0, 42.0)
Y_RANGE = (9.0, 31.0)
Z_FIXED = 0.0


def sample_positions(n, seed):
    """Muestreo uniforme en el rectángulo de alcance. Semilla fija => reproducible."""
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        x, y = rng.uniform(*X_RANGE), rng.uniform(*Y_RANGE)
        # Zonas sin contacto observadas en el piloto (el poste queda fuera del alcance de la matriz derecha):
        if x > 33 and y > 27:      # esquina lejana
            continue
        if x < 14 and y < 11:      # rincón junto a la cabeza
            continue
        out.append((float(x), float(y), Z_FIXED))
    return out


def run_one(sim_id, pos, out_dir, t_stop):
    """Ejecuta una simulación. Devuelve (sim_id, estado, segundos)."""
    d = os.path.join(out_dir, f"sim_{sim_id:04d}")
    done_flag = os.path.join(d, "DONE")
    if os.path.exists(done_flag):
        return sim_id, "skip", 0.0
    os.makedirs(d, exist_ok=True)
    x, y, z = pos
    cmd = [BIN, "--MODEL_TYPE", "0", "--DEBUG", "0", "--PRINT", "0",
           "--OBJECT", "1", "--SPEED", "0", "--PEG_LOC", f"{x:.3f}", f"{y:.3f}", f"{z:.3f}",
           "--TIME_STOP", f"{t_stop}", "--ACTIVE", "1", "--WHISKER_NAMES", "R",
           "--SAVE", "1", "--SAVE_VIDEO", "0", "--dir_out", d]
    t0 = time.time()
    with open(os.path.join(d, "whiskit.log"), "w") as log:
        rc = subprocess.call(cmd, cwd=SCRIPTS, stdout=log, stderr=subprocess.STDOUT)
    dt = time.time() - t0
    if rc == 0 and os.path.exists(os.path.join(d, "dynamics", "Mz.csv")):
        open(done_flag, "w").write(f"{dt:.1f}\n")
        return sim_id, "ok", dt
    return sim_id, f"fail(rc={rc})", dt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=40)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="pilot_A")
    ap.add_argument("--time", type=float, default=1.0, help="segundos simulados por corrida")
    ap.add_argument("--jobs", type=int, default=2)
    a = ap.parse_args()

    out_dir = os.path.join(HERE, a.out)
    os.makedirs(out_dir, exist_ok=True)
    positions = sample_positions(a.n, a.seed)

    # Manifiesto: una fila por simulación con su etiqueta. Se escribe ANTES de correr,
    # así la etiqueta nunca depende de que la simulación termine.
    man_path = os.path.join(out_dir, "manifest.csv")
    with open(man_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["sim_id", "peg_x_mm", "peg_y_mm", "peg_z_mm", "t_stop_s", "object", "whiskers", "seed"])
        for i, (x, y, z) in enumerate(positions):
            w.writerow([i, f"{x:.3f}", f"{y:.3f}", f"{z:.3f}", a.time, "peg_static", "R", a.seed])

    print(f"[dataset] {a.n} simulaciones -> {out_dir}  (jobs={a.jobs}, t={a.time}s)", flush=True)
    t_start = time.time()
    with ThreadPoolExecutor(max_workers=a.jobs) as ex:
        futs = [ex.submit(run_one, i, p, out_dir, a.time) for i, p in enumerate(positions)]
        for k, fut in enumerate(futs, 1):
            sim_id, st, dt = fut.result()
            el = (time.time() - t_start) / 60
            print(f"[{k:3d}/{a.n}] sim_{sim_id:04d} {st:10s} {dt:6.1f}s  (transcurrido {el:5.1f} min)", flush=True)
    print("[dataset] listo", flush=True)


if __name__ == "__main__":
    main()
''')
open('/content/dataset/build_npz.py','w').write(r'''#!/usr/bin/env python3
"""
Empaqueta las simulaciones de WHISKiT en un único archivo .npz listo para entrenar.

Forma de salida (Tarea A, localización):
   X : (N, T, W, C)  float32   N simulaciones, T pasos de 1 ms, W=27 bigotes, C=6 canales [Mx,My,Mz,Fx,Fy,Fz]
   y : (N, 2)        float32   posición del poste (x, y) en mm, marco de la cabeza
   meta: sim_id, whisker_names, channel_names, t_stop, dropped_initial_steps

Decisiones de diseño (y por qué):
 - El eje de secuencia es el TIEMPO, no el índice de bigote. Esto corrige la inversión de ejes del TFM.
 - Se descartan los primeros `drop` pasos: el simulador produce un transitorio numérico (~1e16) al inicializar.
 - NO se normaliza aquí. La normalización se ajusta SOLO con el conjunto de entrenamiento, en el script de
   entrenamiento, para evitar fuga de información del test.
 - Si una simulación tiene NaN (inestabilidad numérica) se registra y se excluye; nunca se rellena con ceros
   en silencio.

Uso:
   python3 build_npz.py --dir pilot_A --out pilot_A.npz
"""
import argparse, csv, os, json
import numpy as np
import pandas as pd

CHANNELS = ["Mx", "My", "Mz", "Fx", "Fy", "Fz"]


def read_matrix(path, n_whiskers):
    # Los CSV de WHISKiT terminan cada fila en coma -> columna vacía final; nos quedamos con las W primeras.
    return pd.read_csv(path, header=None).values[:, :n_whiskers].astype(np.float32)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--drop", type=int, default=5, help="pasos iniciales a descartar (transitorio)")
    a = ap.parse_args()

    base = os.path.join(os.path.dirname(os.path.abspath(__file__)), a.dir)
    man = list(csv.DictReader(open(os.path.join(base, "manifest.csv"))))

    X, y, ids, dropped = [], [], [], []
    whisker_names = None
    T_ref = None
    for row in man:
        sid = int(row["sim_id"])
        d = os.path.join(base, f"sim_{sid:04d}")
        if not os.path.exists(os.path.join(d, "DONE")):
            dropped.append((sid, "no terminada"))
            continue
        if whisker_names is None:
            whisker_names = [s.strip() for s in open(os.path.join(d, "whisker_ID.csv")).read().split(",") if s.strip()]
        W = len(whisker_names)
        mats = [read_matrix(os.path.join(d, "dynamics", f"{c}.csv"), W) for c in CHANNELS]
        arr = np.stack(mats, axis=-1)[a.drop:]            # (T, W, C)
        if np.isnan(arr).any() or np.abs(arr).max() > 1e6:
            dropped.append((sid, "NaN o explosión numérica"))
            continue
        if T_ref is None:
            T_ref = arr.shape[0]
        if arr.shape[0] != T_ref:                           # longitudes distintas -> recortar al mínimo común
            T_ref = min(T_ref, arr.shape[0])
        X.append(arr)
        y.append([float(row["peg_x_mm"]), float(row["peg_y_mm"])])
        ids.append(sid)

    X = np.stack([x[:T_ref] for x in X]).astype(np.float32)
    y = np.array(y, dtype=np.float32)
    np.savez_compressed(os.path.join(os.path.dirname(base), a.out), X=X, y=y, sim_id=np.array(ids),
                        whisker_names=np.array(whisker_names), channel_names=np.array(CHANNELS),
                        dropped_initial_steps=a.drop)
    print(f"X {X.shape}  y {y.shape}  guardado en {a.out}")
    print(f"rango |M| canales 0-2: {np.abs(X[..., :3]).max():.3g}   rango |F| canales 3-5: {np.abs(X[..., 3:]).max():.3g}")
    if dropped:
        print("excluidas:", dropped)
    json.dump({"n": len(ids), "T": int(T_ref), "W": int(X.shape[2]), "C": int(X.shape[3]),
               "dropped": dropped}, open(os.path.join(base, "build_report.json"), "w"), indent=1)


if __name__ == "__main__":
    main()
''')
print('scripts escritos')

In [ ]:
# El generador asume la estructura <raíz>/whiskitphysics/code y <raíz>/dataset; en Colab la raíz es /content.
# Las simulaciones se guardan en Drive (WORK) mediante un enlace simbólico para que sobrevivan a la desconexión.
import os
link = '/content/dataset/full_A'
if not os.path.islink(link):
    os.makedirs(f'{WORK}/full_A', exist_ok=True)
    os.symlink(f'{WORK}/full_A', link)
print('salida ->', os.path.realpath(link))

## 3. Generar el dataset (reanudable)
Ajusta `--n` (número de simulaciones) y `--jobs` (núcleos). La semilla fija garantiza que las posiciones sean las mismas en cada reanudación.

In [ ]:
import os
jobs = os.cpu_count()
!cd /content/dataset && python3 run_dataset.py --n 300 --seed 1 --out full_A --time 1.0 --jobs {jobs}

## 4. Empaquetar en un .npz y guardarlo en Drive

In [ ]:
!cd /content/dataset && python3 build_npz.py --dir full_A --out dataset_A.npz
import shutil; shutil.copy('/content/dataset/dataset_A.npz', f'{WORK}/dataset_A.npz')
import numpy as np; d = np.load(f'{WORK}/dataset_A.npz')
print('X', d['X'].shape, ' y', d['y'].shape, ' bigotes', list(d['whisker_names']))